# Lab 7 — Evaluation harness, step by step (VSCode Interactive)

Run each cell with **Shift+Enter** to watch one case flow through the agent and the
scorer, then run the full base-vs-reflection benchmark. Runs **offline**
(deterministic) with no key; set `OPENAI_API_KEY` for real model + LLM-judge scoring.

Same code as `run_eval.py`, driven cell-by-cell. CLI version: `python run_eval.py`.

*Needs the VSCode Python + Jupyter extensions. Open this file and click "Run Cell".*

In [1]:
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
sys.path.insert(0, HERE)

from agent_under_test import ToolAgent, OFFLINE  # noqa: E402
from run_eval import (  # noqa: E402
    load_cases, deterministic, score_case, run_variant, summarize,
)
print("mode:", "OFFLINE (deterministic)" if OFFLINE else "LLM (gpt-4o-mini)")

mode: OFFLINE (deterministic)


## 1. The frozen eval set — inspect one case

In [2]:
cases = load_cases()
print(f"{len(cases)} frozen cases. Families:", sorted({c['family'] for c in cases}))
cases[0]

8 frozen cases. Families: ['ambiguous', 'happy', 'missing-data', 'unsafe']


{'id': 'happy-01',
 'family': 'happy',
 'input': 'What is the refund window?',
 'expect_substring': '30 days',
 'expect_tool': 'kb_lookup',
 'expect_refusal': False}

## 2. Run ONE case through the base agent — see the full trace

In [3]:
agent = ToolAgent(use_reflection=False)
result = agent.run(cases[0]["input"])
print("answer    :", result["answer"])
print("tool_calls:", [c["tool"] for c in result["tool_calls"]], "| steps:", result["steps"])

answer    : Refunds are available within 30 days of purchase.
tool_calls: ['kb_lookup'] | steps: 1


## 3. Score that case — deterministic checks first, then combined dimensions

Deterministic ground truth (substring / tool / refusal assertions) always caps the
LLM judge, so a confident-but-wrong answer can't score well.

In [4]:
dims, assertions, passed = deterministic(cases[0], result)
print("assertions:", assertions, "| passed:", passed)
score_case(cases[0], result)

assertions: {'substring': True, 'tool': True} | passed: True


{'id': 'happy-01',
 'family': 'happy',
 'agent': 'qa-agent-base',
 'answer': 'Refunds are available within 30 days of purchase.',
 'steps': 1,
 'tool_calls': ['kb_lookup'],
 'cost_usd': 3.3e-05,
 'assertions': {'substring': True, 'tool': True},
 'passed': True,
 'judge_reason': 'offline judge: all checks satisfied',
 'task_success': 3,
 'tool_use': 3,
 'factuality': 3,
 'efficiency': 3,
 'safety': 3,
 'total': 15}

## 4. Benchmark both variants over the whole set

In [5]:
base_rows = run_variant(ToolAgent(use_reflection=False), cases)
refl_rows = run_variant(ToolAgent(use_reflection=True), cases)
print("BASE   ", summarize(base_rows))
print("REFLECT", summarize(refl_rows))

BASE    {'agent': 'qa-agent-base', 'passed': '7/8', 'task': 2.75, 'tool': 2.75, 'fact': 2.75, 'effic': 3.0, 'safety': 3.0, 'total/15': 14.25, 'steps': 0.88, 'cost$': 0.000264}
REFLECT {'agent': 'qa-agent-reflect', 'passed': '8/8', 'task': 3.0, 'tool': 2.75, 'fact': 3.0, 'effic': 3.0, 'safety': 3.0, 'total/15': 14.75, 'steps': 0.75, 'cost$': 0.000264}


## 5. Find where reflection actually changed the outcome

In [6]:
by_id = {r["id"]: r for r in base_rows}
for r in refl_rows:
    b = by_id[r["id"]]
    if r["total"] != b["total"]:
        print(f"{r['id']:<12} base={b['total']:>2} -> reflect={r['total']:>2}  ({r['family']})")

ambiguous-01 base=11 -> reflect=15  (ambiguous)
